In [ ]:
import numpy as np
import rasterio
from tqdm import tqdm
import os
import gc

# === Paths ===
disturbance_folder = r"G:\Hangkai\CONUS_Forest_Edge_LCMAP\1km_dominant_forest_disturbance_most_recent"
area_folder = r"G:\Hangkai\CONUS_Forest_Edge_LCMAP\Forest_landscape_dynamics_Output"
edge_folder = area_folder
years = list(range(1989, 2022))  # Change relative to previous year

# === Reference TIF to get shape/profile ===
ref_tif = os.path.join(area_folder, "Forest_Area_2001_1km.tif")
with rasterio.open(ref_tif) as src:
    height, width = src.height, src.width
    profile = src.profile

# === Initialize arrays to accumulate change per disturbance (0–8) ===
area_contrib = np.zeros((9, height, width), dtype=np.float32)
edge_contrib = np.zeros((9, height, width), dtype=np.float32)

# === Loop through years ===
for year in tqdm(years):
    try:
        area_cur = rasterio.open(os.path.join(area_folder, f"Forest_Area_{year}_1km.tif")).read(1)
        area_prev = rasterio.open(os.path.join(area_folder, f"Forest_Area_{year - 1}_1km.tif")).read(1)
        area_change = (area_cur.astype(np.int32) - area_prev.astype(np.int32)) / 900  # unit: pixels
        del area_cur, area_prev
        gc.collect()

        edge_cur = rasterio.open(os.path.join(edge_folder, f"Edge_Length_{year}_1km.tif")).read(1)
        edge_prev = rasterio.open(os.path.join(edge_folder, f"Edge_Length_{year - 1}_1km.tif")).read(1)
        edge_change = (edge_cur.astype(np.int32) - edge_prev.astype(np.int32)) / 30  # unit: pixels
        del edge_cur, edge_prev
        gc.collect()

        disturbance_path = os.path.join(disturbance_folder, f"Most_Recent_Disturbance_Summary_{year - 1}.tif")
        if not os.path.exists(disturbance_path):
            print(f"⚠️ Missing disturbance file for year {year-1}")
            continue
        disturbance = rasterio.open(disturbance_path).read(1)

        for d in range(9):
            mask = (disturbance == d)
            area_contrib[d][mask] += area_change[mask]
            edge_contrib[d][mask] += edge_change[mask]

        del disturbance, mask, area_change, edge_change
        gc.collect()

    except Exception as e:
        print(f"❌ Error in year {year}: {e}")
        continue

# === Calculate dominant disturbance by absolute change ===
area_major_disturb_net = np.argmax(np.abs(area_contrib), axis=0).astype(np.uint8)
edge_major_disturb_net = np.argmax(np.abs(edge_contrib), axis=0).astype(np.uint8)

# === Save output ===
profile.update({"dtype": "uint8", "count": 1, "compress": "lzw"})
with rasterio.open(os.path.join(area_folder, "DominantDisturbance_AreaChange_NET_1988_2021.tif"), "w", **profile) as dst:
    dst.write(area_major_disturb_net, 1)
with rasterio.open(os.path.join(edge_folder, "DominantDisturbance_EdgeChange_NET_1988_2021.tif"), "w", **profile) as dst:
    dst.write(edge_major_disturb_net, 1)

print("✅ Net change (positive/negative) dominant disturbance maps saved.")